In [ ]:
!pip install -q langchain langchain-openai langgraph pydantic

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'voc-130443061316695052884116a9085edaa1529.73089617'

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, List, Optional

class UserIntent(BaseModel):
    """Classifies what the user is asking for."""
    intent_type: Literal["qa", "summarize", "calculate"] = Field(
        ..., description="The classified intent of the user's request"
    )
    confidence: float = Field(
        ..., ge=0.0, le=1.0, description="Confidence score between 0 and 1"
    )
    reasoning: str = Field(
        ..., description="Brief explanation for why this intent was chosen"
    )

class AnswerResponse(BaseModel):
    """The final structured response returned to the user."""
    answer: str = Field(..., description="The main response content")
    confidence: float = Field(
        default=0.8, ge=0.0, le=1.0, description="Confidence in the answer"
    )
    sources: List[str] = Field(
        default_factory=list, description="Document sections or sources used"
    )
    tool_calls_made: List[str] = Field(
        default_factory=list, description="Names of tools invoked to produce this answer"
    )

# --- quick validation smoke test ---
try:
    UserIntent(intent_type="qa", confidence=1.5, reasoning="test")
except Exception as e:
    print("Validation correctly rejected confidence=1.5:", e)

good = UserIntent(intent_type="calculate", confidence=0.92, reasoning="Contains a math expression")
print(good)

Validation correctly rejected confidence=1.5: 1 validation error for UserIntent
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
intent_type='calculate' confidence=0.92 reasoning='Contains a math expression'


In [ ]:
import ast
import operator
import json
import os
from datetime import datetime
from langchain_core.tools import tool

os.makedirs("logs", exist_ok=True)

_ALLOWED_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
    ast.Mod: operator.mod,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")

def _log_tool_call(expression: str, result: str, success: bool):
    entry = {
        "timestamp": datetime.utcnow().isoformat(),
        "tool": "calculator",
        "expression": expression,
        "result": result,
        "success": success,
    }
    with open("logs/tool_calls.jsonl", "a") as f:
        f.write(json.dumps(entry) + "\n")

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely and return the result as a string.
    Supports +, -, *, /, **, %, and parentheses. Example: '(120 + 45) * 3'."""
    try:
        parsed = ast.parse(expression, mode="eval")
        result = _safe_eval(parsed.body)
        result_str = str(result)
        _log_tool_call(expression, result_str, success=True)
        return result_str
    except Exception as e:
        error_msg = f"Error evaluating expression: {e}"
        _log_tool_call(expression, error_msg, success=False)
        return error_msg

# quick test
print(calculator.invoke({"expression": "(120 + 45) * 3"}))
print(calculator.invoke({"expression": "import os"}))  # should fail safely

495
Error evaluating expression: invalid syntax (<unknown>, line 1)


/tmp/ipykernel_18619/912514733.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

INTENT_CLASSIFICATION_PROMPT = """You are an intent classifier for a document assistant.
Classify the user's request into exactly one of these categories:

- "qa": The user is asking a specific factual question about document content
  (e.g. "What was the total revenue in Q3?", "Who is the patient's primary physician?")
- "summarize": The user wants an overview, summary, or key points extracted
  (e.g. "Summarize this report", "What are the main takeaways?")
- "calculate": The user wants a mathematical operation performed on numbers in the document
  (e.g. "What's the sum of all expenses?", "Calculate the percentage increase")

Respond with the intent_type, a confidence score (0-1) reflecting how certain you are,
and a brief one-sentence reasoning for your classification.

User request: {user_input}
"""

def get_chat_prompt_template(intent_type: str) -> ChatPromptTemplate:
    """Returns the appropriate system+human prompt template based on classified intent."""
    system_messages = {
        "qa": (
            "You are a precise document Q&A assistant. Answer only using information "
            "found in the provided document context. If the answer isn't in the document, "
            "say so clearly rather than guessing. Cite the relevant section when possible."
        ),
        "summarize": (
            "You are a concise summarization assistant. Produce a clear, well-organized "
            "summary of the document, highlighting key points, figures, and conclusions. "
            "Keep it shorter than the original while preserving essential meaning."
        ),
        "calculate": (
            "You are a careful calculation assistant. Identify the relevant numbers in the "
            "document, use the calculator tool to compute results precisely, and show your "
            "reasoning. Never guess at arithmetic — always use the tool."
        ),
    }
    system_msg = system_messages.get(intent_type, system_messages["qa"])
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "Document context:\n{document_context}\n\nUser request: {user_input}"),
    ])

In [ ]:
from typing import TypedDict, List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://openai.vocareum.com/v1",
)

structured_intent_llm = llm.with_structured_output(UserIntent)
structured_answer_llm = llm.with_structured_output(AnswerResponse)

class AgentState(TypedDict):
    user_input: str
    document_context: str
    intent: Optional[UserIntent]
    tool_calls_made: List[str]
    final_response: Optional[AnswerResponse]

def classify_intent_node(state: AgentState) -> AgentState:
    prompt = INTENT_CLASSIFICATION_PROMPT.format(user_input=state["user_input"])
    intent = structured_intent_llm.invoke(prompt)
    state["intent"] = intent
    return state

def qa_node(state: AgentState) -> AgentState:
    template = get_chat_prompt_template("qa")
    messages = template.format_messages(
        document_context=state["document_context"], user_input=state["user_input"]
    )
    response = structured_answer_llm.invoke(messages)
    response.sources = ["document_context"]
    state["final_response"] = response
    return state

def summarize_node(state: AgentState) -> AgentState:
    template = get_chat_prompt_template("summarize")
    messages = template.format_messages(
        document_context=state["document_context"], user_input=state["user_input"]
    )
    response = structured_answer_llm.invoke(messages)
    response.sources = ["document_context"]
    state["final_response"] = response
    return state

def calculate_node(state: AgentState) -> AgentState:
    template = get_chat_prompt_template("calculate")
    llm_with_tools = llm.bind_tools([calculator])
    messages = template.format_messages(
        document_context=state["document_context"], user_input=state["user_input"]
    )
    ai_msg = llm_with_tools.invoke(messages)

    tool_calls_made = []
    tool_results = []
    for tc in getattr(ai_msg, "tool_calls", []):
        result = calculator.invoke(tc["args"])
        tool_calls_made.append(f"calculator({tc['args'].get('expression')}) = {result}")
        tool_results.append(result)

    answer_text = ai_msg.content if ai_msg.content else "; ".join(tool_results)
    state["final_response"] = AnswerResponse(
        answer=answer_text or "No calculation could be performed.",
        confidence=0.95 if tool_results else 0.4,
        sources=["document_context"],
        tool_calls_made=tool_calls_made,
    )
    return state

def route_by_intent(state: AgentState) -> str:
    return state["intent"].intent_type

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

def create_workflow():
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_intent", classify_intent_node)
    workflow.add_node("qa", qa_node)
    workflow.add_node("summarize", summarize_node)
    workflow.add_node("calculate", calculate_node)

    workflow.set_entry_point("classify_intent")

    workflow.add_conditional_edges(
        "classify_intent",
        route_by_intent,
        {"qa": "qa", "summarize": "summarize", "calculate": "calculate"},
    )

    workflow.add_edge("qa", END)
    workflow.add_edge("summarize", END)
    workflow.add_edge("calculate", END)

    memory = MemorySaver()
    return workflow.compile(checkpointer=memory)

app = create_workflow()

In [ ]:
import uuid, json, os
from datetime import datetime

os.makedirs("sessions", exist_ok=True)

class DocumentAssistant:
    def __init__(self, workflow):
        self.workflow = workflow

    def ask(self, user_input: str, document_context: str, session_id: str = None) -> AnswerResponse:
        session_id = session_id or str(uuid.uuid4())
        config = {"configurable": {"thread_id": session_id}}

        state = {
            "user_input": user_input,
            "document_context": document_context,
            "intent": None,
            "tool_calls_made": [],
            "final_response": None,
        }
        result = self.workflow.invoke(state, config=config)
        response = result["final_response"]

        self._log_session(session_id, user_input, result["intent"], response)
        return response, session_id

    def _log_session(self, session_id, user_input, intent, response):
        log_path = f"sessions/{session_id}.jsonl"
        entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "user_input": user_input,
            "intent": intent.model_dump() if intent else None,
            "response": response.model_dump(),
        }
        with open(log_path, "a") as f:
            f.write(json.dumps(entry) + "\n")

assistant = DocumentAssistant(app)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://openai.vocareum.com/v1",  # confirm this from your project instructions
)

In [ ]:
sample_doc = """
Q3 Financial Report: Total revenue was $450,000, up from $380,000 in Q2.
Operating expenses were $210,000. Net profit margin improved due to reduced
marketing spend. The healthcare division reported 1,200 new patient visits.
"""

response, sid = assistant.ask("What was the total revenue in Q3?", sample_doc)
print("Q&A:", response, "\n")

response, sid = assistant.ask("Summarize this report in two sentences.", sample_doc)
print("Summary:", response, "\n")

response, sid = assistant.ask("Calculate the increase from Q2 to Q3 revenue.", sample_doc)
print("Calculation:", response)

/tmp/ipykernel_18619/43262638.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Q&A: answer='The total revenue in Q3 was $450,000.' confidence=0.95 sources=['document_context'] tool_calls_made=[] 



/tmp/ipykernel_18619/43262638.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Summary: answer='In Q3, total revenue increased to $450,000 from $380,000 in Q2, with operating expenses at $210,000, leading to an improved net profit margin due to reduced marketing spend. The healthcare division also saw growth with 1,200 new patient visits.' confidence=0.95 sources=['document_context'] tool_calls_made=[] 

Calculation: answer='70000' confidence=0.95 sources=['document_context'] tool_calls_made=['calculator(450000 - 380000) = 70000']


/tmp/ipykernel_18619/912514733.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
